# 1 - Data Acquisition

In [242]:
using Graphs
using MatrixDepot
using SimpleWeightedGraphs
using Random

## 1.1 - Zachary's Karate Club Network
The `karate` dataset is a classic social network graph consisting of 34 nodes and 78 edges. It is natively included within the `Graphs.jl` ecosystem.

In [243]:
karate_graph = smallgraph(:karate)

println("Karate network loaded successfully:")
println("- Number of nodes: ", nv(karate_graph))
println("- Number of edges: ", ne(karate_graph))

Karate network loaded successfully:
- Number of nodes: 34
- Number of edges: 78


## 1.2 - The Dolphins Social Network
The `dolphins` dataset contains 62 nodes and 159 edges. We use `MatrixDepot.jl` to automatically fetch the sparse adjacency matrix from the SuiteSparse/DIMACS public archive (indexed as "Newman/dolphins") and convert it into a standard undirected graph.

In [244]:
dolphins_matrix = matrixdepot("Newman/dolphins")
dolphins_graph = SimpleGraph(dolphins_matrix)

println("\nDolphins network loaded successfully:")
println("- Number of nodes: ", nv(dolphins_graph))
println("- Number of edges: ", ne(dolphins_graph))


Dolphins network loaded successfully:
- Number of nodes: 62
- Number of edges: 159


## 1.3 - Political Books Network
The political books dataset contains 105 books about US politics and the edges indicate frequent co-purchasing by the same buyers. We load it from MatrixDepot as an undirected graph.

In [245]:
polbooks_matrix = matrixdepot("Newman/polbooks")
polbooks_graph = SimpleGraph(polbooks_matrix)

println("Political Books network loaded successfully:")
println("- Number of nodes: ", nv(polbooks_graph))
println("- Number of edges: ", ne(polbooks_graph))

Political Books network loaded successfully:
- Number of nodes: 105
- Number of edges: 441


## 1.4 - College Football Network
The college football dataset models the schedule of games between 115 Division IA college football teams. It is a standard benchmark for community detection.

In [246]:
football_matrix = matrixdepot("Newman/football")
football_graph = SimpleGraph(football_matrix)

println("College Football network loaded successfully:")
println("- Number of nodes: ", nv(football_graph))
println("- Number of edges: ", ne(football_graph))

College Football network loaded successfully:
- Number of nodes: 115
- Number of edges: 613


# 2 - Objective Function & Baseline Heuristic


## 2.1 - Objective function 

To compare candidate partitions for modularity maximization, we compute Newman-Girvan modularity.
This score measures how much more edge weight lies inside communities than would be expected by chance in a random graph with the same node strengths.

The implementation below is generic:
- unweighted graphs are treated as if every existing edge has weight `1.0`
- weighted graphs built with `SimpleWeightedGraphs.jl` use their actual edge weights
- the community assignment is given as a label vector indexed by vertex

This lets us evaluate both `Graph` and `SimpleWeightedGraph` inputs using the same objective function.

In [247]:
function modularity(g, community)
    n = nv(g)
    @assert length(community) == n "community vector must have one label per vertex"

    has_weight = hasmethod(weight, Tuple{typeof(g), Int, Int})
    edge_weight(u, v) = has_weight ? weight(g, u, v) : 1.0

    m = 0.0
    degree_weight = zeros(Float64, n)
    for e in edges(g)
        u, v = src(e), dst(e)
        w = edge_weight(u, v)
        m += w
        degree_weight[u] += w
        degree_weight[v] += w
    end

    if m == 0.0
        return 0.0
    end

    two_m = 2.0 * m

    community_degree = Dict{Int, Float64}()
    for u in 1:n
        c = community[u]
        community_degree[c] = get(community_degree, c, 0.0) + degree_weight[u]
    end

    intra_weight = 0.0
    for e in edges(g)
        u, v = src(e), dst(e)
        if community[u] == community[v]
            intra_weight += edge_weight(u, v)
        end
    end

    Q = intra_weight / m
    for (_, Dt) in community_degree
        Q -= (Dt / two_m)^2
    end

    return Q
end

modularity (generic function with 1 method)

In [248]:
# Sanity check: modularity of the trivial (one-community)
# We should obtain 0 because we didn't compute the clusters of the network yet.
function whole_graph_modularity(g)
    trivial_community = ones(Int, nv(g))
    return modularity(g, trivial_community)
end

whole_graph_modularity (generic function with 1 method)

In [249]:
println("Dolphins whole-graph modularity: ", whole_graph_modularity(dolphins_graph))
println("Karate whole-graph modularity: ", whole_graph_modularity(karate_graph))

Dolphins whole-graph modularity: 0.0
Karate whole-graph modularity: 0.0


## 2.2 - Baseline Heuristic


### 2.2.1 - Simple Label Propagation (LPA)

We implement a basic Label Propagation Algorithm inspired by Section 3 of the LPAm+ paper.
The procedure is intentionally simple:
1. every node is initially assigned a unique label;
2. at each iteration, each node adopts the most frequent label among its neighbors;
3. when there is a tie, one of the tied labels is chosen uniformly at random.

This version does not include modularity maximization or community merging, so the result can be unstable and may yield one or two communities depending on the random seed.

In [250]:
function lpa(g; max_iter=100, seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)

    labels = collect(1:n)
    current_labels = copy(labels)

    for _ in 1:max_iter
        changed = false
        new_labels = copy(current_labels)

        for u in 1:n
            nbrs = [v for v in neighbors(g, u) if v != 0]
            if isempty(nbrs)
                continue
            end

            counts = Dict{Int, Int}()
            for v in nbrs
                lab = current_labels[v]
                counts[lab] = get(counts, lab, 0) + 1
            end

            max_count = maximum(values(counts))
            candidates = [lab for (lab, count) in counts if count == max_count]
            chosen_label = candidates[rand(rng, 1:length(candidates))]

            if new_labels[u] != chosen_label
                new_labels[u] = chosen_label
                changed = true
            end
        end

        current_labels = new_labels
        if !changed
            break
        end
    end

    return current_labels
end


lpa (generic function with 1 method)

In [251]:
karate_labels = lpa(karate_graph; seed=42)
dolphins_labels = lpa(dolphins_graph; seed=42)

println("Karate labels: ", karate_labels)
println("Number of karate communities: ", length(unique(karate_labels)))
println("Dolphins labels: ", dolphins_labels)
println("Number of dolphins communities: ", length(unique(dolphins_labels)))

Karate labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 15, 1, 1, 1, 1, 15, 15, 1, 1, 15, 1, 15, 1, 15, 15, 1, 15, 15, 15, 1, 15, 1, 15, 1, 1]
Number of karate communities: 2
Dolphins labels: [31, 10, 31, 22, 22, 6, 6, 10, 22, 10, 31, 22, 19, 10, 19, 22, 19, 6, 22, 10, 19, 22, 10, 22, 22, 10, 10, 10, 31, 22, 31, 10, 6, 19, 19, 22, 19, 19, 19, 6, 19, 6, 31, 19, 19, 22, 19, 31, 6, 19, 19, 22, 19, 19, 10, 22, 10, 10, 19, 22, 10, 19]
Number of dolphins communities: 5


### 2.2.2 - Label Propagation with Modularity Maximization (LPAm)

We now refine the label propagation result by using the modularity objective as a local optimization criterion.
Starting from the simple LPA partition, each node is revisited and moved to the neighboring community (or a new singleton community) that yields the highest modularity gain.
When several moves give the same modularity score, one of them is selected at random.

In [252]:
function initialize_label_state(g, initial_labels, degrees)
    n = nv(g)
    mapping = Dict{Int, Int}()
    next_id = 1
    for lab in unique(initial_labels)
        mapping[lab] = next_id
        next_id += 1
    end
    current_labels = [mapping[lab] for lab in initial_labels]

    max_possible_labels = n
    D = zeros(Int, max_possible_labels)
    nodes_in_label = [Set{Int}() for _ in 1:max_possible_labels]

    for u in 1:n
        lab = current_labels[u]
        push!(nodes_in_label[lab], u)
        D[lab] += degrees[u]
    end

    active_labels = Set(unique(current_labels))
    return current_labels, D, nodes_in_label, active_labels
end

function neighbor_label_counts(g, u, current_labels)
    label_counts = Dict{Int, Int}()
    for v in neighbors(g, u)
        if v != 0
            lab_v = current_labels[v]
            label_counts[lab_v] = get(label_counts, lab_v, 0) + 1
        end
    end
    return label_counts
end

function ensure_label_exists!(D, nodes_in_label)
    unused_label = findfirst(==(0), D)
    if unused_label === nothing
        push!(D, 0)
        push!(nodes_in_label, Set{Int}())
        unused_label = length(D)
    end
    return unused_label
end

function choose_best_label!(u, current_lab, D, nodes_in_label, degrees, two_m, label_counts, rng)
    best_score = -Inf
    best_label = current_lab

    for (cand, k_u_to_cand) in label_counts
        D_l = D[cand]
        score = k_u_to_cand - (degrees[u] * D_l) / two_m

        if score > best_score + 1e-12
            best_score = score
            best_label = cand
        elseif abs(score - best_score) <= 1e-12 && rand(rng) < 0.5
            best_label = cand
        end
    end

    unused_label = ensure_label_exists!(D, nodes_in_label)
    score_unused = 0.0 - (degrees[u] * D[unused_label]) / two_m

    if score_unused > best_score + 1e-12
        best_score = score_unused
        best_label = unused_label
    elseif abs(score_unused - best_score) <= 1e-12 && rand(rng) < 0.5
        best_label = unused_label
    end

    return best_label
end

function lpam(g; max_iter=20, seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)
    two_m = 2.0 * ne(g)
    degrees = degree(g)

    initial_labels = lpa(g; seed=seed)
    current_labels, D, nodes_in_label, active_labels = initialize_label_state(g, initial_labels, degrees)

    iter = 0
    while !isempty(active_labels) && iter < max_iter
        iter += 1

        lab = pop!(active_labels)
        if isempty(nodes_in_label[lab])
            continue
        end

        nodes_to_check = collect(nodes_in_label[lab])
        shuffle!(rng, nodes_to_check)

        for u in nodes_to_check
            if current_labels[u] != lab
                continue
            end

            current_lab = current_labels[u]
            D[current_lab] -= degrees[u]

            label_counts = neighbor_label_counts(g, u, current_labels)
            best_label = choose_best_label!(u, current_lab, D, nodes_in_label, degrees, two_m, label_counts, rng)

            D[best_label] += degrees[u]

            if best_label != current_lab
                delete!(nodes_in_label[current_lab], u)
                push!(nodes_in_label[best_label], u)
                current_labels[u] = best_label

                push!(active_labels, current_lab)
                push!(active_labels, best_label)

                for v in neighbors(g, u)
                    push!(active_labels, current_labels[v])
                end
            end
        end
    end

    return current_labels
end


lpam (generic function with 1 method)

In [253]:
karate_lpam_labels = lpam(karate_graph; seed=42)
dolphins_lpam_labels = lpam(dolphins_graph; seed=42)

println("Karate LPAm labels: ", karate_lpam_labels)
println("Number of karate communities: ", length(unique(karate_lpam_labels)))
println("Karate LPAm modularity: ", modularity(karate_graph, karate_lpam_labels))
println("\nDolphins LPAm labels: ", dolphins_lpam_labels)
println("Number of dolphins communities: ", length(unique(dolphins_lpam_labels)))
println("Dolphins LPAm modularity: ", modularity(dolphins_graph, dolphins_lpam_labels))

Karate LPAm labels: [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Number of karate communities: 2
Karate LPAm modularity: 0.3717948717948718

Dolphins LPAm labels: [1, 2, 1, 3, 3, 2, 2, 2, 3, 2, 1, 3, 5, 2, 5, 3, 5, 2, 3, 2, 5, 3, 2, 3, 3, 2, 2, 2, 1, 3, 1, 2, 2, 5, 5, 3, 5, 5, 5, 2, 5, 2, 1, 5, 5, 3, 5, 1, 2, 5, 5, 3, 5, 5, 2, 3, 2, 2, 5, 3, 2, 5]
Number of dolphins communities: 4
Dolphins LPAm modularity: 0.5264625608164234


### 2.2.3 - Community Merging (LPAm+)



Compared with LPAm, LPAm+ adds a second refinement step in which neighboring communities are greedily merged whenever that increases modularity. This helps avoid over-fragmentation and often yields a more stable partition than the local-move phase alone.

The implementation below follows the paper’s idea of starting from an LPAm partition and then applying a merge phase driven by the modularity objective. First, here are the helper functions.


In [254]:
function relabel_communities(labels)
    mapping = Dict{Int, Int}()
    next_id = 1
    relabeled = copy(labels)

    for i in eachindex(relabeled)
        lab = relabeled[i]
        if !haskey(mapping, lab)
            mapping[lab] = next_id
            next_id += 1
        end
        relabeled[i] = mapping[lab]
    end

    return relabeled
end

function communities_are_adjacent(g, labels, a, b)
    for u in 1:nv(g)
        if labels[u] != a
            continue
        end

        for v in neighbors(g, u)
            if v != 0 && labels[v] == b
                return true
            end
        end
    end

    return false
end

function merge_gain(g, labels, a, b)
    merged_labels = [lab == b ? a : lab for lab in labels]
    return modularity(g, merged_labels) - modularity(g, labels)
end

function best_merge_pair(g, labels)
    best_gain = 0.0
    best_pair = nothing

    communities = unique(labels)
    for i in 1:length(communities)-1
        for j in i+1:length(communities)
            a = communities[i]
            b = communities[j]

            if !communities_are_adjacent(g, labels, a, b)
                continue
            end

            gain = merge_gain(g, labels, a, b)
            if gain > best_gain + 1e-12
                best_gain = gain
                best_pair = (a, b)
            end
        end
    end

    return best_pair, best_gain
end

function merge_communities!(g, labels)
    current_labels = relabel_communities(labels)

    while true
        best_pair, best_gain = best_merge_pair(g, current_labels)
        if best_pair === nothing || best_gain <= 1e-12
            break
        end

        a, b = best_pair
        current_labels = [lab == b ? a : lab for lab in current_labels]
        current_labels = relabel_communities(current_labels)
    end

    return current_labels
end

function lpam_with_initial_labels(g, initial_labels; max_iter=20, seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)
    two_m = 2.0 * ne(g)
    degrees = degree(g)

    current_labels, D, nodes_in_label, active_labels = initialize_label_state(g, initial_labels, degrees)

    iter = 0
    while !isempty(active_labels) && iter < max_iter
        iter += 1

        lab = pop!(active_labels)
        if isempty(nodes_in_label[lab])
            continue
        end

        nodes_to_check = collect(nodes_in_label[lab])
        shuffle!(rng, nodes_to_check)

        for u in nodes_to_check
            if current_labels[u] != lab
                continue
            end

            current_lab = current_labels[u]
            D[current_lab] -= degrees[u]

            label_counts = neighbor_label_counts(g, u, current_labels)
            best_label = choose_best_label!(u, current_lab, D, nodes_in_label, degrees, two_m, label_counts, rng)

            D[best_label] += degrees[u]

            if best_label != current_lab
                delete!(nodes_in_label[current_lab], u)
                push!(nodes_in_label[best_label], u)
                current_labels[u] = best_label

                push!(active_labels, current_lab)
                push!(active_labels, best_label)

                for v in neighbors(g, u)
                    push!(active_labels, current_labels[v])
                end
            end
        end
    end

    return current_labels
end

lpam_with_initial_labels (generic function with 1 method)

In [255]:
function lpam_plus(g; max_iter=20, seed=1)
    initial_labels = lpa(g; seed=seed)
    current_labels = lpam_with_initial_labels(g, initial_labels; max_iter=max_iter, seed=seed)

    while true
        n_communities_before = length(unique(current_labels))

        merged_labels = merge_communities!(g, current_labels)

        if length(unique(merged_labels)) == n_communities_before
            current_labels = merged_labels
            break
        end

        current_labels = lpam_with_initial_labels(g, merged_labels; max_iter=max_iter, seed=seed)
    end

    return current_labels
end

lpam_plus (generic function with 1 method)

In [256]:
karate_lpam_plus_labels = lpam_plus(karate_graph; seed=42)
dolphins_lpam_plus_labels = lpam_plus(dolphins_graph; seed=42)

println("Karate LPAm+ labels: ", karate_lpam_plus_labels)
println("Number of karate communities: ", length(unique(karate_lpam_plus_labels)))
println("Karate LPAm+ modularity: ", modularity(karate_graph, karate_lpam_plus_labels))
println("\nDolphins LPAm+ labels: ", dolphins_lpam_plus_labels)
println("Number of dolphins communities: ", length(unique(dolphins_lpam_plus_labels)))
println("Dolphins LPAm+ modularity: ", modularity(dolphins_graph, dolphins_lpam_plus_labels))

Karate LPAm+ labels: [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Number of karate communities: 2
Karate LPAm+ modularity: 0.3717948717948718

Dolphins LPAm+ labels: [1, 2, 1, 3, 3, 2, 2, 2, 3, 2, 1, 3, 4, 2, 4, 3, 4, 2, 3, 2, 4, 3, 2, 3, 3, 2, 2, 2, 1, 3, 1, 2, 2, 4, 4, 3, 4, 4, 4, 2, 4, 2, 1, 4, 4, 3, 4, 1, 2, 4, 4, 3, 4, 4, 2, 3, 2, 2, 4, 3, 2, 4]
Number of dolphins communities: 4
Dolphins LPAm+ modularity: 0.5264625608164234
